In [ ]:
import actionet
import scipy
import numpy as np
import scanpy as sc
import anndata
import pandas as pd

import lets_plot
lets_plot.LetsPlot.setup_html()

In [ ]:
X = scipy.io.mmread("../data/counts.mtx")
obs_df = pd.read_csv("../data/sample_data.tsv", sep="\t", header=0)
var_df = pd.read_csv("../data/feature_data.tsv", sep="\t", header=0)

adata = anndata.AnnData(
    X=scipy.sparse.csr_matrix(X).T,
    var=var_df,
    obs=obs_df
)

In [ ]:
actionet.filter_anndata(adata, min_cells_per_feat=0.01)
adata.layers['logcounts'] = sc.pp.normalize_total(adata, target_sum=1e4, inplace=False)['X']
sc.pp.log1p(adata, layer='logcounts', base=2, copy=False)
actionet.reduce_kernel(adata, n_components=30, layer='logcounts', key_added='action')

In [ ]:
actionet.run_actionet(adata, k_max=30, layer='logcounts', layout_epochs=150)

In [ ]:
actionet.plot_umap(adata, color='CellLabel')

In [ ]:
adata.write_h5ad("../data/test_adata_post.h5ad")